# CT4110 – Topic 10 Exercise
## From a symmetric QUBO to QAOA in Qiskit

This exercise develops a consistent mathematical formulation of number partitioning and carries it all the way into a working QAOA implementation.

You will learn to:

1. formulate number partitioning as
   $$
   f(\mathbf{x})=\mathbf{x}^{T}Q\mathbf{x}+\mathbf{v}^{T}\mathbf{x}+c,
   $$
   where $Q$ is symmetric;
2. absorb the linear term into the diagonal to obtain a symmetric matrix $Q'$;
3. convert the QUBO into the Ising cost Hamiltonian $H_C$;
4. implement $U_C(\gamma)=e^{-i\gamma H_C}$ and the mixer
   $U_M(\beta)=e^{-i\beta H_M}$ in Qiskit;
5. use a classical optimiser to choose QAOA angles;
6. compare QAOA with exact enumeration and a classical heuristic.

In [ ]:
# Run this cell first.

import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "qiskit": "qiskit>=2.2",
    "qiskit_optimization": "qiskit-optimization>=0.7",
}

missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

from itertools import product

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.translators import to_ising

np.set_printoptions(precision=3, suppress=True)

## 1. Number partitioning as a symmetric QUBO

Let the positive weights be $w_0,\ldots,w_{n-1}$, and let

$$
W=\sum_{i=0}^{n-1}w_i.
$$

The binary variable $x_i$ records the group containing item $i$:

$$
x_i=
\begin{cases}
1,&\text{item }i\text{ is in group A},\\
0,&\text{item }i\text{ is in group B}.
\end{cases}
$$

The two group totals are

$$
S_A=\sum_i w_i x_i,
\qquad
S_B=W-S_A.
$$

We minimise the squared imbalance

$$
f(\mathbf{x})=(S_A-S_B)^2=(2\mathbf{w}^{T}\mathbf{x}-W)^2.
$$

Expanding gives

$$
f(\mathbf{x})
=\mathbf{x}^{T}Q\mathbf{x}+\mathbf{v}^{T}\mathbf{x}+c,
$$

with

$$
Q_{ij}=4w_iw_j,
\qquad
v_i=-4Ww_i,
\qquad
c=W^2.
$$

Because $x_i^2=x_i$ for binary variables, the linear term can be absorbed into the diagonal:

$$
Q'_{ij}=Q_{ij}+v_i\delta_{ij}
       =4\left(w_iw_j-Ww_i\delta_{ij}\right).
$$

Therefore,

$$
\boxed{f(\mathbf{x})=\mathbf{x}^{T}Q'\mathbf{x}+c}
$$

with $Q'$ symmetric.

Expanding this symmetric quadratic form once more gives

$$
\mathbf{x}^{T}Q'\mathbf{x}
=
\sum_i Q'_{ii}x_i
+
2\sum_{i<j}Q'_{ij}x_ix_j.
$$

The coefficient of the monomial $x_ix_j$ is therefore $2Q'_{ij}$, because the matrix entries $Q'_{ij}$ and $Q'_{ji}$ both contribute.

## 2. From the symmetric QUBO to an Ising Hamiltonian

Use the binary-to-spin substitution

$$
x_i=\frac{1-z_i}{2},
\qquad z_i\in\{+1,-1\}.
$$

For

$$
f(\mathbf{x})=\mathbf{x}^{T}Q'\mathbf{x}+c,
$$

the Ising energy is

$$
E(\mathbf{z})
=
\kappa+\sum_i h_i z_i+\sum_{i<j}J_{ij}z_iz_j.
$$

Substitution and collection of constant, linear, and quadratic spin terms gives

$$
\boxed{
h_i=-\frac12\sum_j Q'_{ij},
\qquad
J_{ij}=\frac12Q'_{ij},
\qquad
\kappa=c+\frac14\mathbf{1}^{T}Q'\mathbf{1}
              +\frac14\operatorname{Tr}(Q').
}
$$

### Why some QUBO formulas contain a factor of $1/4$

The same symmetric QUBO can be written as a sum of monomials,

$$
f(\mathbf{x})
=
c+\sum_i a_ix_i+\sum_{i<j}b_{ij}x_ix_j,
$$

where

$$
a_i=Q'_{ii},
\qquad
b_{ij}=2Q'_{ij}.
$$

In that notation,

$$
J_{ij}=\frac{b_{ij}}{4}=\frac{Q'_{ij}}{2},
$$

$$
h_i=-\frac{a_i}{2}
    -\frac14\sum_{j\ne i}b_{\min(i,j),\max(i,j)}
    =-\frac12\sum_jQ'_{ij},
$$

and

$$
\kappa
=
c+\frac12\sum_i a_i+\frac14\sum_{i<j}b_{ij}
=
c+\frac14\mathbf{1}^{T}Q'\mathbf{1}
 +\frac14\operatorname{Tr}(Q').
$$

The two sets of formulas are identical; their coefficients look different only because $b_{ij}$ is the full coefficient of the monomial $x_ix_j$, while $Q'_{ij}$ is one of two equal off-diagonal matrix entries.

### Promote the Ising energy to an operator

Replace each spin value by a Pauli-$Z$ operator:

$$
z_i\longrightarrow Z_i.
$$

The cost Hamiltonian is

$$
\boxed{
H_C=\kappa I+\sum_i h_iZ_i+\sum_{i<j}J_{ij}Z_iZ_j.
}
$$

For every computational-basis state $|\mathbf{x}\rangle$,

$$
H_C|\mathbf{x}\rangle=f(\mathbf{x})|\mathbf{x}\rangle.
$$

Thus the basis states are eigenstates of $H_C$, and their eigenvalues are their QUBO scores.

### Number-partitioning simplification

For the absorbed matrix,

$$
\sum_jQ'_{ij}
=
4\left(w_i\sum_jw_j-Ww_i\right)=0.
$$

Consequently, every $h_i$ vanishes. Moreover,

$$
J_{ij}=2w_iw_j,
\qquad
\kappa=\sum_iw_i^2.
$$

Hence

$$
\boxed{
H_C=
\left(\sum_iw_i^2\right)I
+
2\sum_{i<j}w_iw_jZ_iZ_j.
}
$$

This result provides a direct check of the general conversion formulas.

## 3. Hamiltonians, unitaries, and the QAOA loop

A Hamiltonian is **Hermitian**; it is not generally unitary. The circuit uses its exponential:

$$
U_C(\gamma)=e^{-i\gamma H_C}.
$$

If

$$
H_C=\kappa I+\widetilde H_C,
$$

then

$$
U_C(\gamma)
=e^{-i\gamma\kappa}e^{-i\gamma\widetilde H_C}.
$$

The identity term contributes the global phase $e^{-i\gamma\kappa}$. It shifts every energy by $\kappa$ but cannot change measurement probabilities. We may therefore evolve with $\widetilde H_C$ and add $\kappa$ back when reporting the QUBO energy.

The standard mixer is

$$
H_M=\sum_i X_i,
\qquad
U_M(\beta)=e^{-i\beta H_M}
           =\bigotimes_i R_X(2\beta).
$$

A depth-$p$ QAOA state is

$$
|\psi_p(\boldsymbol\gamma,\boldsymbol\beta)\rangle
=
U_M(\beta_p)U_C(\gamma_p)\cdots
U_M(\beta_1)U_C(\gamma_1)
|+\rangle^{\otimes n}.
$$

Here

$$
\boldsymbol\gamma=(\gamma_1,\ldots,\gamma_p),
\qquad
\boldsymbol\beta=(\beta_1,\ldots,\beta_p).
$$

### Step 1: encode cost information as relative phase

Using the full Hamiltonian,

$$
U_C(\gamma)|\mathbf{x}\rangle
=
e^{-i\gamma f(\mathbf{x})}|\mathbf{x}\rangle.
$$

If the identity term is omitted from the circuit, the phase is instead
$e^{-i\gamma(f(\mathbf{x})-\kappa)}$. The two descriptions differ only by a common global phase, so all relative phases are unchanged.

### Step 2: convert relative phases into amplitudes

The mixer Hamiltonian $H_M$ connects computational-basis states at Hamming distance one. Its exponential $U_M(\beta)$ mixes amplitudes across this hypercube of bit strings. Depending on $\gamma$ and $\beta$, paths into a bit string can interfere constructively or destructively.

The mixer does **not** identify low-energy states by itself.

### Step 3: use classical feedback

The classical optimiser changes the angles to minimise

$$
F_p(\boldsymbol\gamma,\boldsymbol\beta)
=
\langle\psi_p(\boldsymbol\gamma,\boldsymbol\beta)|
H_C
|\psi_p(\boldsymbol\gamma,\boldsymbol\beta)\rangle
=
\sum_{\mathbf{x}}
P_{\boldsymbol\gamma,\boldsymbol\beta}(\mathbf{x})f(\mathbf{x}).
$$

Minimising this weighted average encourages probability to move toward lower-energy bit strings. It does not guarantee that every measurement is optimal.

QAOA is a structured variational algorithm because both alternating operators reflect the optimisation problem: $H_C$ contains the objective values and $H_M$ defines movement through the search space. This structure does not guarantee an advantage over a generic VQE ansatz or a classical method.

## 4. Implementing the two unitaries in Qiskit

Represent the non-identity part $\widetilde H_C$ with a `SparsePauliOp`. Then

```python
qc.append(PauliEvolutionGate(cost_operator, time=gamma), range(n))
```

implements

$$
e^{-i\gamma\widetilde H_C}.
$$

All cost terms in this exercise contain only $Z$ operators, so they commute.

The same cost unitary can be implemented term by term:

```python
qc.rz(2 * gamma * h_i, i)
qc.rzz(2 * gamma * J_ij, i, j)
```

because

$$
R_Z(\theta)=e^{-i\theta Z/2},
\qquad
R_{ZZ}(\theta)=e^{-i\theta Z\otimes Z/2}.
$$

Implement the mixer with one $X$ rotation per qubit:

```python
for qubit in range(n):
    qc.rx(2 * beta, qubit)
```

### Qiskit ordering

- Qubit $i$ represents variable $x_i$.
- Qiskit displays bit strings as $q_{n-1}\cdots q_0$.
- Reverse a displayed key before interpreting it as $(x_0,\ldots,x_{n-1})$.
- Pauli labels use the same right-to-left qubit ordering.

---
# Task 1 — Easy: two-variable partition

Begin with the two weights

$$
\mathbf{w}=(5,9),
\qquad W=14.
$$

For this problem,

$$
Q=
\begin{pmatrix}
100&180\\
180&324
\end{pmatrix},
\qquad
\mathbf{v}=
\begin{pmatrix}
-280\\-504
\end{pmatrix},
\qquad
c=196.
$$

After absorbing $\mathbf{v}$ into the diagonal,

$$
Q'=
\begin{pmatrix}
-180&180\\
180&-180
\end{pmatrix}.
$$

You will check the four binary assignments classically, derive $H_C$, and implement one QAOA layer in two equivalent ways.

In [ ]:
# Task 1.1 — classical reference level
#
# Instructions
# 1. Define partition_score(bits, weights).
# 2. Return (S_A, S_B, imbalance, squared_imbalance).
# 3. Enumerate 00, 01, 10, and 11.
# 4. Store the records in easy_results and print them.
# 5. Identify every assignment with the minimum squared imbalance.

easy_weights = np.array([5, 9], dtype=float)

def partition_score(bits, weights):
    # YOUR CODE HERE
    raise NotImplementedError

easy_results = []
# YOUR CODE HERE

In [ ]:
# Task 1.2 — verify the symmetric QUBO
#
# Instructions
# 1. Construct easy_Q, easy_v, easy_Qp, and easy_c from the formulas above.
# 2. Check that Q and Qp are symmetric.
# 3. Define qubo_score(bits, Qp, constant) = bits.T @ Qp @ bits + constant.
# 4. For all four bit strings, compare qubo_score with the classical squared imbalance.
# 5. Assert that every pair agrees numerically.
#
# Expected absorbed matrix:
# [[-180,  180],
#  [ 180, -180]]

easy_Q = None
easy_v = None
easy_Qp = None
easy_c = None

def qubo_score(bits, Qp, constant):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

In [ ]:
# Task 1.3 — derive and build the cost Hamiltonian
#
# For a symmetric absorbed matrix Qp:
#   h_i   = -0.5 * sum_j Qp[i, j]
#   J_ij  =  0.5 * Qp[i, j]       for i < j
#   kappa = c + 0.25 * 1.T @ Qp @ 1 + 0.25 * trace(Qp)
#
# Instructions
# 1. Calculate easy_h, easy_J, and easy_kappa.
# 2. Confirm that easy_h = [0, 0], easy_J[(0, 1)] = 90, and kappa = 106.
# 3. Build the identity-free operator H_tilde = 90 Z_0 Z_1 as a SparsePauliOp.
#    For two qubits the Pauli label is "ZZ".
# 4. Build the full operator H_C = 106 I + H_tilde.
# 5. For every basis state, compare its H_C eigenvalue with qubo_score.
#
# Hint:
#   Statevector.from_label("01").expectation_value(operator)
# uses the displayed order |q_1 q_0>. Convert carefully from (x_0, x_1).

easy_h = None
easy_J = {}
easy_kappa = None
easy_cost_evolution = None
easy_cost_full = None

# YOUR CODE HERE

In [ ]:
# Task 1.3b — represent the same QUBO with Qiskit Optimization
#
# Instructions
# 1. Create QuadraticProgram("two_item_partition").
# 2. Add binary variables x0 and x1, in that order.
# 3. Call minimize(constant=easy_c, quadratic=easy_Qp). Passing the full
#    symmetric array preserves the expression x.T @ Qp @ x.
# 4. Evaluate qp.objective on all four bit strings and compare with qubo_score.
# 5. Call to_ising(qp). It returns (operator_without_identity, offset).
# 6. Confirm that the returned offset is easy_kappa and that adding
#    offset*I produces the same full Hamiltonian as easy_cost_full.
#
# This checkpoint connects the mathematical notation, Qiskit's optimisation model,
# and the SparsePauliOp that is evolved by the QAOA circuit.

easy_qp = None
easy_qiskit_operator = None
easy_qiskit_offset = None

# YOUR CODE HERE

In [ ]:
# Task 1.4 — implement U_C and U_M in two equivalent ways
#
# Use gamma = 0.31 and beta = -0.22 for this circuit-equivalence check.
#
# Circuit A:
#   1. Apply H to both qubits to prepare |++>.
#   2. Append PauliEvolutionGate(easy_cost_evolution, time=gamma).
#   3. Apply RX(2*beta) to both qubits.
#
# Circuit B:
#   1. Apply H to both qubits.
#   2. Apply RZZ(2*gamma*J_01) to qubits (0, 1).
#   3. Apply RX(2*beta) to both qubits.
#
# Compare the two statevectors with .equiv(...). They should agree up to global phase.
# Draw one circuit with output="mpl".

gamma = 0.31
beta = -0.22

easy_qaoa_evolution = QuantumCircuit(2)
easy_qaoa_direct = QuantumCircuit(2)

# YOUR CODE HERE

In [ ]:
# Task 1.5 — let a classical optimiser choose the p=1 angles
#
# Instructions
# 1. Define build_qaoa(cost_evolution, gammas, betas).
#    - infer n from cost_evolution.num_qubits
#    - require len(gammas) == len(betas)
#    - prepare |+>^n
#    - alternate PauliEvolutionGate(cost_evolution, time=gamma_l)
#      and RX(2*beta_l) on every qubit
# 2. Define qaoa_energy(parameters, cost_evolution, cost_full, p).
#    - first p entries are gammas; the remaining p entries are betas
#    - construct the unmeasured circuit
#    - use Statevector.from_instruction(circuit)
#    - return the real expectation value of cost_full
# 3. Minimise the p=1 objective with scipy.optimize.minimize(method="COBYLA").
# 4. Confirm that the best expectation approaches the exact optimum, 16.
#
# Note: the identity term is absent from the evolution circuit but present in
# cost_full, so the reported objective remains on the original QUBO scale.

def build_qaoa(cost_evolution, gammas, betas):
    # YOUR CODE HERE
    raise NotImplementedError

def qaoa_energy(parameters, cost_evolution, cost_full, p):
    # YOUR CODE HERE
    raise NotImplementedError

easy_initial = np.array([0.2, -0.2])
easy_optimisation = None

# YOUR CODE HERE

In [ ]:
# Task 1.6 — inspect the final probability distribution
#
# Instructions
# 1. Build the p=1 circuit using the optimised angles.
# 2. Obtain its Statevector.
# 3. Use probabilities_dict() to obtain exact basis-state probabilities.
# 4. Plot them with plot_histogram.
# 5. Reverse each displayed key before treating it as (x_0, x_1).
# 6. Calculate the total probability of the optimal assignments.
#
# The optimiser minimises an expectation, not a single measured bit string.
# A final run must therefore be sampled and the low-cost strings identified.

easy_final_state = None
easy_probabilities = None
easy_optimal_probability = None

# YOUR CODE HERE

### Task 1 written check

Explain, in your own words:

1. why $H_C$ is Hermitian rather than unitary;
2. why $e^{-i\gamma H_C}$ is unitary;
3. what the identity term $106I$ changes in the energy and what it does **not** change in the circuit's measurement probabilities;
4. why the cost unitary alone cannot change the initially uniform probabilities;
5. how the mixer can turn cost-dependent phases into amplitude differences.

In [ ]:
# YOUR ANSWERS HERE

---
# Task 2 — Medium: four-variable vehicle load balancing

A depot must divide four indivisible loads between two vehicles:

$$
\mathbf{w}=(8,7,6,5).
$$

The binary variable $x_i=1$ assigns load $i$ to vehicle A; $x_i=0$ assigns it to vehicle B.

You will now generalise every step:

1. obtain the exact classical reference;
2. construct the symmetric $Q'$;
3. convert any symmetric $Q'$ into a Qiskit cost operator;
4. create and optimise a reusable QAOA circuit;
5. decode the most probable assignments and compare them with the exact answer.

In [ ]:
# Task 2.1 — exact classical reference and symmetric QUBO
#
# Instructions
# 1. Store medium_weights = [8, 7, 6, 5].
# 2. Enumerate all 16 assignments and find all exact optima.
# 3. Define partition_qubo(weights), returning (Q, v, Qp, c) using:
#       Q  = 4 * outer(w, w)
#       v  = -4 * W * w
#       Qp = Q + diag(v)
#       c  = W**2
# 4. Confirm that Qp is symmetric and every row sums to zero.
# 5. Confirm qubo_score(bits, Qp, c) matches the classical score for all 16 strings.

medium_weights = np.array([8, 7, 6, 5], dtype=float)

def partition_qubo(weights):
    # YOUR CODE HERE
    raise NotImplementedError

medium_exact_results = []
medium_optimal_bits = []
medium_Q = medium_v = medium_Qp = medium_c = None

# YOUR CODE HERE

In [ ]:
# Task 2.2 — generic symmetric-QUBO to Ising conversion
#
# Complete symmetric_qubo_to_ising(Qp, constant).
#
# It must return:
#   cost_full       = kappa*I + sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
#   cost_evolution  = the same operator without kappa*I
#   kappa, h, J
#
# Qiskit Pauli labels are ordered q_(n-1)...q_0. To place Z on qubit i:
#   label = ["I"] * n
#   label[n - 1 - i] = "Z"
#
# Skip coefficients whose absolute value is below atol. Finish each operator
# with .simplify().
#
# Verification:
# 1. Convert the medium QUBO.
# 2. Check h is zero for this partitioning problem.
# 3. For every basis state, compare <x|H_C|x> with qubo_score.
# 4. Check kappa equals sum_i w_i**2.

def symmetric_qubo_to_ising(Qp, constant, atol=1e-12):
    # YOUR CODE HERE
    raise NotImplementedError

medium_cost_full = None
medium_cost_evolution = None
medium_kappa = None
medium_h = None
medium_J = None

# YOUR CODE HERE

In [ ]:
# Task 2.3 — optimise a p=1 QAOA circuit
#
# Instructions
# 1. Reuse build_qaoa and qaoa_energy from Task 1.
# 2. Use several starting points because a variational objective may have
#    multiple local minima.
# 3. Run COBYLA from each starting point.
# 4. Keep the result with the lowest objective.
# 5. Compare the QAOA expectation with the exact minimum.
# 6. Draw the final circuit.
#
# Suggested starts are supplied, but you may add more.

medium_starts = [
    np.array([0.1, 0.1]),
    np.array([0.5, -0.3]),
    np.array([1.0, 0.4]),
    np.array([1.5, -0.7]),
]

medium_runs = []
medium_best = None

# YOUR CODE HERE

In [ ]:
# Task 2.4 — decode probabilities and evaluate solution quality
#
# Instructions
# 1. Build the circuit at medium_best.x and obtain probabilities_dict().
# 2. Sort the outcomes from most to least probable.
# 3. For each of the eight most probable keys:
#       bits = np.array([int(b) for b in key[::-1]])
#    Then report probability, bits, vehicle totals, and QUBO score.
# 4. Calculate:
#       a. probability of sampling an exact optimum;
#       b. expected score computed directly from the probability dictionary.
# 5. Confirm (b) agrees with medium_best.fun.
# 6. Plot the distribution.

medium_final_state = None
medium_probabilities = None
medium_optimal_probability = None

# YOUR CODE HERE

### Task 2 written check

Discuss:

1. why complementary strings $\mathbf{x}$ and $\mathbf{1}-\mathbf{x}$ represent the same physical split with vehicle labels exchanged;
2. why this symmetry produces degenerate ground states;
3. why the classical optimiser changes $\gamma$ and $\beta$, rather than changing the Hamiltonian;
4. whether a low expectation value guarantees that every measurement is optimal;
5. why QAOA is more structured than an arbitrary hardware-efficient VQE ansatz, while still offering no automatic performance advantage.

In [ ]:
# YOUR ANSWERS HERE

---
# Advanced Task 3 — Hard: six variables, deeper QAOA, and a heuristic solver

Two vans must share six indivisible containers:

$$
\mathbf{w}=(3,5,6,7,9,11).
$$

The total is odd, so a zero imbalance is impossible. Your goals are to:

1. build the six-variable symmetric QUBO and exact benchmark;
2. compare $p=1$ and $p=2$ QAOA;
3. implement a one-bit-flip hill climber with random restarts;
4. compare exact search, QAOA, and the heuristic honestly.

For $p=2$, the circuit is

$$
U_M(\beta_2)U_C(\gamma_2)
U_M(\beta_1)U_C(\gamma_1)|+\rangle^{\otimes 6}.
$$

Increasing $p$ increases expressivity, but also enlarges the classical parameter search.

In [ ]:
# Task 3.1 — exact benchmark, QUBO, and Hamiltonian
#
# Instructions
# 1. Enumerate all 64 assignments.
# 2. Find the exact minimum and all optimal bit strings.
# 3. Construct hard_Q, hard_v, hard_Qp, and hard_c.
# 4. Convert them with symmetric_qubo_to_ising.
# 5. Verify all 64 Hamiltonian eigenvalues against the classical/QUBO scores.
# 6. Print the number of Pauli terms in the identity-free cost operator.

hard_weights = np.array([3, 5, 6, 7, 9, 11], dtype=float)

hard_exact_results = []
hard_optimal_bits = []
hard_Q = hard_v = hard_Qp = hard_c = None
hard_cost_full = hard_cost_evolution = None

# YOUR CODE HERE

In [ ]:
# Task 3.2 — compare p=1 and p=2 QAOA
#
# Instructions
# 1. For p=1, optimise [gamma_1, beta_1] from at least four starts.
# 2. For p=2, optimise [gamma_1, gamma_2, beta_1, beta_2] from at least
#    eight starts. Use a seeded RNG to make the starts reproducible.
# 3. Record for each depth:
#       - best expected QUBO score;
#       - probability of sampling an exact optimum;
#       - the most probable decoded bit string and its score;
#       - number of objective evaluations.
# 4. Do not assume p=2 must win: report what your optimisation actually finds.
#
# Suggested optimiser settings:
#   method="COBYLA", options={"maxiter": 250}

rng = np.random.default_rng(4110)
qaoa_comparison = {}

# YOUR CODE HERE

In [ ]:
# Task 3.3 — implement one-bit-flip hill climbing
#
# Instructions
# 1. Define one_bit_neighbours(bits). For n bits it must return n neighbours,
#    each differing in exactly one position.
# 2. Define hill_climb(start_bits, Qp, constant).
# 3. At each step, move to the strictly best lower-scoring neighbour.
# 4. Stop if no neighbour has a lower score.
# 5. Return the final bits, final score, and the full history.
# 6. Test the function from at least three chosen starting strings.
#
# Use qubo_score as the objective so the heuristic and QAOA solve exactly the
# same mathematical problem.

def one_bit_neighbours(bits):
    # YOUR CODE HERE
    raise NotImplementedError

def hill_climb(start_bits, Qp, constant):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

In [ ]:
# Task 3.4 — random-restart heuristic
#
# Instructions
# 1. Define random_restart_hill_climb(Qp, constant, restarts, seed).
# 2. Generate each initial bit string with np.random.default_rng(seed).
# 3. Run hill_climb from every start.
# 4. Return the best result and all individual run records.
# 5. Test restart budgets [1, 5, 20, 100].
# 6. For each budget, repeat the complete experiment over 20 different seeds.
# 7. Report:
#       - exact-optimum success rate;
#       - mean best score;
#       - worst best score;
#       - mean number of hill-climb moves.

def random_restart_hill_climb(Qp, constant, restarts, seed):
    # YOUR CODE HERE
    raise NotImplementedError

restart_budgets = [1, 5, 20, 100]
heuristic_summary = {}

# YOUR CODE HERE

In [ ]:
# Task 3.5 — final comparison
#
# Build a concise comparison table with one row for each method:
#   exact enumeration, QAOA p=1, QAOA p=2, and 100-restart hill climbing.
#
# Suggested columns:
#   method
#   best bit-string score found
#   expected score (QAOA only)
#   exact-optimum sampling/success probability
#   main computational cost
#
# Make clear that:
# - a QAOA expectation is not itself a sampled solution;
# - statevector simulation is a classical calculation used here for learning;
# - the hill-climber's success rate is empirical;
# - exact enumeration is feasible here only because n=6.

# YOUR CODE HERE

### Advanced Task 3 final analysis

Answer the following.

1. Why can a one-bit hill climber stop at a non-global optimum?
2. How do random restarts help, and what do they cost?
3. What changes mathematically when QAOA depth increases from $p=1$ to $p=2$?
4. Did $p=2$ improve the expectation and the ground-state probability in your run? If not, distinguish limited ansatz depth from failed parameter optimisation.
5. Why does the mixer not automatically favour low-energy states?
6. In what sense does the classical optimiser make low-energy states more probable?
7. For this six-variable instance, which method is most practical and why?
8. State one reason QAOA may be attractive for structured QUBOs and one reason a classical heuristic may still be preferable.

In [ ]:
# YOUR ANSWERS HERE